# **MD Simulation Colab - classic MD with DNA without ZEN**

This branch purpose is to simulate the duplexes developped during the Sorbonne iGEM's 2026 team projet PUREGRAIN-ZEN. The goal here is to observe if a duplex remains joined, reorders itself, presents conformational changes or separates itself in a classic MD using water an ions. It was originally made as a notebook working in google colab and was adapted to suit local computational needs. The following notebook gathers all the necessary functions calls needed for analysis

This Notebook is an attempt to adapt and expand upon the work previously done by Alexis Fahy, based on a cloud-based MD workflow, ["Making it Rain"](https://github.com/pablo-arantes/making-it-rain). I also thank Halil Arda for collaborating with me in this endeavor and all other contributors who may appear in the future

Along with this notebook you should also have received a folder containing the architecture and files needed to run or test everything. Keep it as is - or change it accordingly if you're familiar with the concept enough.

We start by setting up a GPU environment to work with. On a free colab account you should be able to get a **T4**

In [ ]:
# Setting GPU is important later down the line - check its availability here
gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0:
  print('Activate the GPU via Runtime > Change runtime type, and relaunch this cell.')
else:
  print(gpu_info)

Setup your drive to work. To keep it simple, You should have installed the folder sent with the notebook at the root of your drive (or any other path if you know how to rewrite path and if you'd rather do it your own way). The function `set_dir()` automatically creates a connection to your drive

If you're running this notebook locally, you may skip this section


In [ ]:
import functions.colab_funcs as colab_funcs
import functions.constants

In [ ]:
colab_funcs.set_dir()

# **Step 1: Building the system**

Now system construction. The original code by Alexis used a series of command lines involving Amber and tleap. For simplified use, I made it so the entire system can immediately be built from OpenMM. That does imply that some options that would have been available if we were going completely raw, such as the types of forcefields, are now more restricted, but that should be more than enough for our needs.

Parameters for building the system (and for the equilibration down the line) were chosen either in respect of experimental conditions in our team Wetlab or parameters recommendations extracted from Alexis’s original work and the source cloud-based MD workflow ["Making it Rain"](https://github.com/pablo-arantes/making-it-rain)

Functions have been defined to:
- Prepare the pdb file: identify and label strands in the pdb file, cleanup file errors by leveraging pdbfixer
- Loading and saving systems and models
- Evaluate box sizes - as simulation box too small would cause periodic image issues

In [ ]:
import functions.system_builder as system_builder
import functions.constants as constants

In [ ]:
system_builder.build_system(constants.input_pdb_ext, constants.Z1_ext_res_folder, fit_box=False)

# **Step 2: Minimize and Balance the system**
The purpose of this operation is to minimize and balance the system.

Like the previous Step, this section aims to be independent, considering the possibility that Colab may crash anytime or exceed GPU hours. To start it, you'll need the system and model files that are generated from step 1.

In order to proceed with equilibration, we defined a `Equilibrator` which takes in a ladder and the equilibration pipeline. The ladder serves as a guide to dictate what kind of transformations should be applied to the system during equilibration

In [ ]:
from functions.equilibration import Equilibrator
import functions.constants as constants

In [ ]:
#The default ladder parameter can be seen by calling the ladder property of the class
Equilibrator().ladder

In [ ]:
#You can also pass a custom made ladder to suit your need from class instanciation alone
Equilibrator([("min", 0, 0)]).ladder

In [ ]:
#Call the equilibrate function of the Equilibrator to start production 
simulation = Equilibrator().equilibrate(constants.Z1_ori_res_folder)

Post equilibration, you can see some first metrics by using the evaluation functions available at **step #4**

# **Step 3: Running production**

Now lies the heart of what we're trying to accomplish, simulating the behavior of the duplex. We are, however limited by one thing: computationnal ressources. In order to deal with this in colab, The functions have been designed to organize a workflow between local colab saves and transfers towards a google drive. This, way, multiple accounts can assist with the processing step

One thing to note about processing and saving files: the frequency at which files have to be updated towards the google drive appears so high that it the i/o can be **bugged**: while attempting direct saves towards a google drive, I noticed that the .xtc file i was exporting (which basically contains what we wish to observe) was not being kept up to date with the colab simulation. The workaround found was to work in the local colab env, then set timers which will help flush data towards the Drive and prevent corrupted saves

## **Running for one production**

In [ ]:
from functions.production import produce
import functions.constants as constants

In [ ]:
produce(constants.Z1_ext_res_folder, save_name="Z1_Z0_test_run_short_ns", ns=0.01, save_ps=1)

## **Running for replicates**

The function presented here has been developped in an attemps to simulate multiple replicates with parallel colab accounts. Given a model, It automatically takes care of equilibrating each replicate and launches a production run. 

For equilibration, an Equilibrator class needs to be instanciated. By default the functions handles one, but you'd need to pass one yourself if you desire an equilibration phase different from the default

In [ ]:
from functions.replicates import run_replicas
from functions.equilibration import Equilibrator
import functions.constants as constants

In [ ]:
run_replicas(constants.Z1_ext_res_folder, 
             local_root=constants.Z1_ext_res_folder + "/replicas",
             save_ps=1, ns=0.01,
             equil=Equilibrator([('min', 200, 0)]),
             cleanup_local=True)

# **Step 4: Analysis**

To analyze your trajectories, you have three functions available to you:
- `analyze_run`: takes a production runs a gives a detailed series of metrics, see the table metric file. This function is also useful if you wish to study the state of a particular system post equilibration 

- `analyze_fraying`: taks a production run and attempts to study fraying effects. Metrics produced detailed in the table metrics file

- `analyze replicas`: leverages both `analyze_run` and `analyze_fraying` to studies batches of replicas together. Additionnal produced results detailed in the table metric file

In [ ]:
from functions.evaluation import analyze_run, analyze_fraying, analyze_replicas
import functions.constants

In [ ]:
analyze_replicas(constants.Z1_ext_res_folder, traj="md")